In [ ]:
#Error analysis (qualitative inspection of misclassifications)

In [1]:
# Cell 1 — Load data and rebuild best model for predictions

import numpy as np
import pandas as pd
from pathlib import Path
from scipy import sparse
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# Paths
DATA_PATH = Path("../data/processed/reviews_preprocessed.csv")
FEATURE_DIR = Path("../data/features")

# Load
df = pd.read_csv(DATA_PATH)
X_tfidf = sparse.load_npz(FEATURE_DIR / "tfidf.npz")
y = np.load(FEATURE_DIR / "labels.npy")

# Same split as Phase 5
idx = np.arange(len(y))
train_idx, test_idx = train_test_split(
    idx, test_size=0.2, stratify=y, random_state=42
)

X_train = X_tfidf[train_idx]
X_test = X_tfidf[test_idx]
y_train = y[train_idx]
y_test = y[test_idx]

# Train best model again
model = LogisticRegression(max_iter=2000, class_weight="balanced", n_jobs=-1)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Loaded and predictions ready.")
print("Test size:", len(y_test))

C:\Users\DELL\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


Loaded and predictions ready.
Test size: 10000


In [2]:
#Identify misclassified samples

import numpy as np

# Boolean masks on test set
fp_mask = (y_test == 0) & (y_pred == 1)   # False Positive
fn_mask = (y_test == 1) & (y_pred == 0)   # False Negative

# Convert test indices back to original df indices
test_original_idx = test_idx

fp_indices = test_original_idx[fp_mask]
fn_indices = test_original_idx[fn_mask]

print("False Positives:", len(fp_indices))
print("False Negatives:", len(fn_indices))

False Positives: 454
False Negatives: 1017


In [3]:
# Inspect False Positives

print("===== FALSE POSITIVES (True=Negative, Pred=Positive) =====\n")

sample_fp = fp_indices[:5]

for idx in sample_fp:
    print("Index:", idx)
    print("True label:", df.loc[idx, "sentiment"])
    print("Review (raw):")
    print(df.loc[idx, "review"][:800])
    print("\n---\n")

===== FALSE POSITIVES (True=Negative, Pred=Positive) =====

Index: 10150
True label: 0
Review (raw):
"Before I got this I had regular periods. I am 5'4 and weigh 100lb, I gained 15lbs on this,but I needed to gain weight anyways.It was very easy to put in. I wasn't on my period at that time so I didn't get my period for the next 3 weeks and I thought this was awesome. A week later I got my period and it was heavy most days and then light but it didn't go away for about 6 months which is usual. I was off my period for about 2 weeks and had spotting during that, but after the 2 weeks I got my period and I had it for I don't even know how long but months straight! I didn't want to get it out but I had no choice and got it out my doctor said this was because of stress and not sleeping but my gyno said this happens to many people"

---

Index: 4752
True label: 0
Review (raw):
"I had my Skyla inserted this morning and let me not scare anymore but understand it is the most simple procedure how

In [4]:
# Inspect False Negatives

print("===== FALSE NEGATIVES (True=Positive, Pred=Negative) =====\n")

sample_fn = fn_indices[:5]

for idx in sample_fn:
    print("Index:", idx)
    print("True label:", df.loc[idx, "sentiment"])
    print("Review (raw):")
    print(df.loc[idx, "review"][:800])
    print("\n---\n")

===== FALSE NEGATIVES (True=Positive, Pred=Negative) =====

Index: 29014
True label: 1
Review (raw):
"I've been taking this medicine for 7 years. I only take the1mg pill when I'm having a panic attack. Sometimes more than other times. For me it is a God sent. That horrible tightness in your chest, and shortness in breath is unbearable. However I can see how someone could get hooked on this drug."

---

Index: 27997
True label: 1
Review (raw):
"I developed severe lower back nerve pain which became almost intolerable and severely limited range of motion could barely walk. I was prescribed Tylenol 3, 2 tablets every 4 hours; the first day it worked great but then next day I was vomiting and became severely constipated, doctor switched me to Celebrex which did little for the pain. Trying the Tylenol with stool softener, drinking more water hoping I don't get nauseated."

---

Index: 21829
True label: 1
Review (raw):
"I had to write a review to sing the highest praises for this medication! 

In [5]:
# Compare text length in errors vs correct predictions

# Identify correct predictions
correct_mask = (y_test == y_pred)
correct_indices = test_original_idx[correct_mask]

# Lengths
df["word_len"] = df["review"].str.split().str.len()

fp_lengths = df.loc[fp_indices, "word_len"]
fn_lengths = df.loc[fn_indices, "word_len"]
correct_lengths = df.loc[correct_indices, "word_len"]

print("Average word length:")
print("Correct:", round(correct_lengths.mean(), 2))
print("False Positives:", round(fp_lengths.mean(), 2))
print("False Negatives:", round(fn_lengths.mean(), 2))

Average word length:
Correct: 83.79
False Positives: 83.0
False Negatives: 87.02
